In [3]:
import json
import random
from pathlib import Path
from collections import Counter


INPUT_JSONL = Path("icl_sentence_demos_6label.jsonl")
OUTPUT_DIR = Path("references")

K = 10
SEED = 42

RARE_OR_WEAK_LABELS = {"ModelArchitecture", "Method", "MLModel"}
CONFUSABLE_PAIRS = [
    {"MLModel", "ModelArchitecture"},
    {"Dataset", "DataSource"},
    {"Task", "Method"},
]


def load_jsonl(path: Path):
    items = []
    with path.open("r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                items.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"Skipping bad JSON at line {line_num}: {e}")
    return items


def write_jsonl(items, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for item in items:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")


def label_set(item):
    if "labels_present" in item and item["labels_present"]:
        return set(item["labels_present"])
    labels = set()
    for ent in item.get("entities", []):
        lbl = ent.get("ner_label")
        if lbl:
            labels.add(lbl)
    return labels


def count_labels(items):
    c = Counter()
    for item in items:
        for lbl in label_set(item):
            c[lbl] += 1
    return c



def hardness_score(item, global_label_freq=None):
    """
    Heuristic hardness score for one sentence demo.
    Higher = harder / more informative.
    """
    labels = label_set(item)
    entities = item.get("entities", [])
    text = item.get("context_text", "")

    score = 0.0

    # 1) More entities => denser sentence => harder
    score += len(entities) * 1.0

    # 2) Multi-label examples are harder/more useful
    score += len(labels) * 2.0

    # 3) Reward rare/weak labels
    score += sum(2.5 for lbl in labels if lbl in RARE_OR_WEAK_LABELS)

    # 4) Reward confusable label combinations
    for pair in CONFUSABLE_PAIRS:
        if pair.issubset(labels):
            score += 3.0

    # 5) Longer entity spans can be boundary-sensitive
    avg_span_len = 0.0
    if entities:
        lengths = [max(1, e["end_char"] - e["start_char"]) for e in entities if "start_char" in e and "end_char" in e]
        if lengths:
            avg_span_len = sum(lengths) / len(lengths)
    score += min(avg_span_len / 10.0, 2.0)

    # 6) Slight reward for longer context
    score += min(len(text.split()) / 20.0, 2.0)

    return score


def sample_random_k(items, k, seed=42):
    rng = random.Random(seed)
    if k > len(items):
        raise ValueError(f"k={k} is larger than the number of available demos ({len(items)}).")
    return rng.sample(items, k)


def sample_hard_k(items, k, seed=42):

    rng = random.Random(seed)
    global_freq = count_labels(items)

    scored = []
    for item in items:
        scored.append((hardness_score(item, global_freq), item))

    scored.sort(key=lambda x: x[0], reverse=True)

    selected = []
    used_doc_counts = Counter()

    for score, item in scored:
        doc_id = item.get("doc_id", "UNKNOWN_DOC")
        if used_doc_counts[doc_id] >= 1:
            continue
        selected.append(item)
        used_doc_counts[doc_id] += 1
        if len(selected) == k:
            return selected

    if len(selected) < k:
        already = {item.get("demo_id") for item in selected}
        remaining = [item for _, item in scored if item.get("demo_id") not in already]
        selected.extend(remaining[: k - len(selected)])

    return selected


def main():
    items = load_jsonl(INPUT_JSONL)
    if not items:
        raise ValueError("No demonstrations were loaded from the input JSONL.")

    print(f"Loaded {len(items)} demonstrations from {INPUT_JSONL}")

    random_k = sample_random_k(items, K, seed=SEED)
    hard_k = sample_hard_k(items, K, seed=SEED)

    random_out = OUTPUT_DIR / f"icl_demos_random_k{K}.jsonl"
    hard_out = OUTPUT_DIR / f"icl_demos_hard_k{K}.jsonl"

    write_jsonl(random_k, random_out)
    write_jsonl(hard_k, hard_out)

    print(f"Wrote random k-shot file to: {random_out}")
    print(f"Wrote hard-example k-shot file to: {hard_out}")

    print("\nRandom selection:")
    for x in random_k:
        print("-", x.get("demo_id"), x.get("labels_present"))

    print("\nHard selection:")
    for x in hard_k:
        print("-", x.get("demo_id"), x.get("labels_present"))


if __name__ == "__main__":
    main()

Loaded 12108 demonstrations from icl_sentence_demos_6label.jsonl
Wrote random k-shot file to: references/icl_demos_random_k10.jsonl
Wrote hard-example k-shot file to: references/icl_demos_hard_k10.jsonl

Random selection:
- 10039_2008_06170.txt::sent::147 ['Method']
- 00014_2010_12421.txt::sent::20 ['Task']
- 00003_2103_00020.txt::sent::187 ['Method']
- 00043_1910_10683.txt::sent::12 ['Method']
- 00037_2104_09864.txt::sent::46 ['Method']
- 00033_2012_14740.txt::sent::141 ['Method', 'ModelArchitecture']
- 00021_1805_12471.txt::sent::28 ['Dataset']
- 10056_2004_05675.txt::sent::148 ['MLModel', 'ModelArchitecture']
- 00012_2102_07033.txt::sent::108 ['DataSource']
- 10045_2206_12705.txt::sent::177 ['Method']

Hard selection:
- 00049_2110_02861.txt::sent::66 ['Dataset', 'MLModel', 'Method', 'ModelArchitecture', 'Task']
- 10003_2105_14444.txt::sent::92 ['MLModel', 'Method', 'ModelArchitecture']
- 00023_1708_00055.txt::sent::76 ['Method', 'ModelArchitecture', 'Task']
- 00012_2102_07033.txt::s